# Cuecard Quality Notebook

Interactive analysis of retrieval quality across pipeline modes.
Run cells sequentially — Cell 1 loads fixtures and model.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve() / "src"))

from fastembed import TextEmbedding
from cuecard.eval import load_fixtures, run_eval, format_eval_report
from cuecard.pipeline import run_pipeline
from cuecard.indexer import build_index
from cuecard.parser import parse_rules

FIXTURES = load_fixtures("eval/fixtures/basic.json")
CORPUS_DIR = "eval/corpora"
MODEL_NAME = "jinaai/jina-embeddings-v2-base-code"
model = TextEmbedding(model_name=MODEL_NAME)
print(f"Loaded {len(FIXTURES)} fixtures")

## Score Distribution Analysis

Run embedding eval with wide recall (top_k=20, threshold=0.0) to see the full score landscape.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

summary = run_eval(FIXTURES, CORPUS_DIR, MODEL_NAME, model=model, top_k=20, threshold=0.0)

# Per-tier recall distribution
tiers = {t.tier: t for t in summary.per_tier}
for tier_name, ts in tiers.items():
    print(f"{tier_name:10s}: recall={ts.mean_recall:.3f} noise={ts.mean_noise_ratio:.3f} n={ts.count}")

# Histogram of per-fixture recall
recalls_by_tier = {}
for fr in summary.per_fixture:
    recalls_by_tier.setdefault(fr.difficulty, []).append(fr.recall_at_k)

fig, axes = plt.subplots(1, len(recalls_by_tier), figsize=(16, 4), sharey=True)
for ax, (tier, vals) in zip(axes, sorted(recalls_by_tier.items())):
    ax.hist(vals, bins=10, edgecolor='black', alpha=0.7)
    ax.set_title(f"{tier} (n={len(vals)})")
    ax.set_xlabel("Recall@20")
    ax.set_ylabel("Count")
plt.suptitle("Per-Fixture Recall Distribution by Tier")
plt.tight_layout()
plt.show()

## Threshold Sweep

Sweep thresholds from 0.05 to 0.60 to find the recall-noise tradeoff.

In [ ]:
thresholds = [0.05 * i for i in range(1, 13)]
results_by_threshold = {}

for t in thresholds:
    s = run_eval(FIXTURES, CORPUS_DIR, MODEL_NAME, model=model, top_k=5, threshold=t)
    results_by_threshold[t] = s
    print(f"t={t:.2f}: recall={s.mean_recall:.3f} precision={s.mean_precision:.3f} noise={s.mean_noise_ratio:.3f}")

# Plot recall vs noise at each threshold
recalls = [results_by_threshold[t].mean_recall for t in thresholds]
noises = [results_by_threshold[t].mean_noise_ratio for t in thresholds]
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(thresholds, recalls, 'b-o', label='Recall')
ax2 = ax1.twinx()
ax2.plot(thresholds, noises, 'r-x', label='Noise')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Recall', color='b')
ax2.set_ylabel('Noise Ratio', color='r')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Threshold vs Recall/Noise')
plt.tight_layout()
plt.show()

## Failure Analysis

Find the 20 worst-performing fixtures by recall.

In [ ]:
# Use the baseline summary (threshold=0.30, top_k=5)
baseline = run_eval(FIXTURES, CORPUS_DIR, MODEL_NAME, model=model, top_k=5, threshold=0.30)

failures = sorted(baseline.per_fixture, key=lambda r: r.recall_at_k)[:20]
for f in failures:
    print(f"\n{f.fixture_id} ({f.difficulty}): recall={f.recall_at_k:.2f}, retrieved={f.retrieved_count}")
    print(f"  Query: {f.query[:80]}...")
    if f.retrieved:
        print(f"  Retrieved: {f.retrieved[:3]}")

## Cross-Encoder Comparison

Compare embedding-only vs cross-encoder reranking.

In [ ]:
emb = run_eval(FIXTURES, CORPUS_DIR, MODEL_NAME, model=model, top_k=5, threshold=0.30)
rerank = run_eval(FIXTURES, CORPUS_DIR, MODEL_NAME, model=model, top_k=5, threshold=0.30, mode="rerank")

print(f"{'Metric':<25s} {'Embedding':>10s} {'Rerank':>10s} {'Delta':>10s}")
print("-" * 55)
for metric in ['mean_recall', 'mean_precision', 'mean_noise_ratio', 'mean_context_waste_ratio', 'negative_silence_rate', 'mean_retrieved_count']:
    e = getattr(emb, metric)
    r = getattr(rerank, metric)
    delta = r - e
    print(f"{metric:<25s} {e:>10.3f} {r:>10.3f} {delta:>+10.3f}")

# Per-tier comparison
print(f"\n{'Tier':<10s} {'Emb R@k':>8s} {'Rnk R@k':>8s} {'Emb Noise':>10s} {'Rnk Noise':>10s}")
print("-" * 50)
emb_tiers = {t.tier: t for t in emb.per_tier}
rnk_tiers = {t.tier: t for t in rerank.per_tier}
for tier in ('easy', 'medium', 'hard', 'negative'):
    et = emb_tiers.get(tier)
    rt = rnk_tiers.get(tier)
    if et and rt:
        print(f"{tier:<10s} {et.mean_recall:>8.3f} {rt.mean_recall:>8.3f} {et.mean_noise_ratio:>10.3f} {rt.mean_noise_ratio:>10.3f}")

## Per-Rule Frequency Analysis

Which rules get retrieved most often? Frequent irrelevant retrievals indicate noise sources.

In [ ]:
from collections import Counter

# Use the wide-recall summary from earlier
rule_freq = Counter()
for fr in summary.per_fixture:
    for rule_text in fr.retrieved:
        rule_freq[rule_text] += 1

print("Top 15 most retrieved rules:")
for rule, count in rule_freq.most_common(15):
    print(f"  {count:3d}x  {rule[:80]}...")

## Negative Query Analysis

Which rules incorrectly match negative queries? These are the primary noise sources.

In [ ]:
neg_fixtures = [f for f in baseline.per_fixture if f.difficulty == "negative"]
neg_noise = Counter()
for nf in neg_fixtures:
    for rule_text in nf.retrieved:
        neg_noise[rule_text] += 1

print(f"Negative queries: {len(neg_fixtures)}")
print(f"Negative queries with results (should be 0): {sum(1 for n in neg_fixtures if n.retrieved_count > 0)}")
print(f"\nRules that match negative queries most (NOISE SOURCES):")
for rule, count in neg_noise.most_common(10):
    print(f"  {count:3d}x  {rule[:80]}...")